# What is LangChain?

**LangChain** is an open-source framework for building applications powered by Large Language Models (LLMs) such as GPT, Claude, Gemini, and open-source models.

It helps developers move beyond simple prompt-response interactions and build structured, production-ready AI applications.

LangChain provides tools and abstractions for:

- Prompt management  
- Connecting to different LLM providers  
- Chaining multiple model calls together  
- Integrating external tools and APIs  
- Building intelligent agents  
- Implementing Retrieval-Augmented Generation (RAG) systems  

In short, LangChain acts as a bridge between LLMs and real-world applications.


# Core Components of LangChain

LangChain is built around several key concepts:

### 1. Models
Interfaces for interacting with chat models and LLM providers.

### 2. Prompts
Templates that structure input to the model dynamically.

### 3. Chains
Sequences of operations where the output of one step becomes the input to the next.

### 4. Agents
LLM-powered systems that can decide which tools to use and take actions autonomously.

### 5. Retrieval (RAG)
Combines LLMs with external data sources to generate more accurate and context-aware responses.

---

### Why Use LangChain?

LangChain simplifies building:
- AI chatbots  
- Document question-answering systems  
- Knowledge assistants  
- Research tools  
- Workflow automation systems  

It enables developers to build scalable, modular, and maintainable LLM-powered applications.


### Expert Knowledge Worker

A question answering agent that is an expert knowledge worker

To be used by employees of Insurellm, an Insurance Tech company

The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

### TODAY:
- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

#### PART A: Divide our documents into chunks

### Chunking in RAG (Document Splitter)

#### What is Chunking?
Chunking is the process of splitting large documents into smaller pieces ("chunks") before generating embeddings in a Retrieval-Augmented Generation (RAG) pipeline.

Each chunk:
- Has its own embedding
- Is stored separately in a vector database
- Can be retrieved independently during search

---

#### Why Chunking is Needed
- LLMs have token limits
- Smaller chunks improve retrieval accuracy
- Prevents mixing unrelated content
- Reduces hallucinations

---

#### Common Chunking Types

#### 1. Fixed-Size Chunking
- Split every N tokens (e.g., 500 tokens)
- Simple but may break sentences

#### 2. Recursive / Smart Chunking
- Split by headings → paragraphs → sentences
- Preserves semantic meaning

#### 3. Overlapping Chunking
- Adjacent chunks share some tokens
- Example:
  - Chunk 1: 0–500
  - Chunk 2: 400–900
- Prevents context loss at boundaries

---

#### Typical Settings
- Chunk size: 300–800 tokens
- Overlap: 10–20%

---

#### RAG Workflow with Chunking
1. Document → Split into chunks  
2. Chunks → Create embeddings  
3. Store in vector DB  
4. Query → Retrieve relevant chunks  
5. Pass chunks to LLM for answer generation


In [ ]:
# Installing the required packages

# !pip install langchain langchain-ollama ollama chromadb langchain-chroma langchain-huggingface

In [ ]:
import os
import shutil
import glob
import tiktoken
import numpy as np
from transformers import AutoTokenizer
from huggingface_hub import login
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [ ]:
OLLAMA_MODEL = "llama3.1"
MODEL = "meta-llama/Llama-3.1-8B"
db_name = "/home/alexender/Desktop/Projects/My_projects/Data/simple_rag_vector_db"
EMBED_MODEL = "nomic-embed-text"

In [ ]:
# How many characters in all the documents?

knowledge_base_path = "/home/alexender/Desktop/Projects/My_projects/Data/knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

In [ ]:
# For using Hugging Face
hf_token = os.getenv("HUGGING_FACE_WRITE_TOKEN")
login(hf_token)

In [ ]:
# How many tokens in all the documents?

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokens = tokenizer.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

In [ ]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("/home/alexender/Desktop/Projects/My_projects/Data/knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

In [ ]:
documents[0]

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

In [ ]:
chunks[100]

#### PART B: Make vectors and store in Chroma

In previous Week, we have set up a Hugging Face account and got an HF_TOKEN and we will be using that here also

# Embedding Models

## What is an Embedding Model?

An **embedding model** converts text (or images, audio, etc.) into numerical representations called **vectors**.

Instead of storing words as plain text, the model transforms them into lists of numbers like:

"The cat sat on the mat"  
→ [0.021, -0.884, 1.204, ..., 0.556]

These numbers capture the *meaning* of the text. Similar texts produce similar vectors.

For example:

- "I love programming"  
- "Coding is fun"  

Their vectors will be mathematically close to each other.

---

## Popular Embedding Models

- text-embedding-3-small (OpenAI)
- text-embedding-3-large (OpenAI)
- all-MiniLM-L6-v2 (SentenceTransformers)

---

## Why Embeddings Matter

Embeddings allow:

- Semantic search
- Clustering
- Recommendation systems
- Retrieval-Augmented Generation (RAG)
- Question answering over documents

In simple terms:

**Embedding model = Converts meaning → numbers**


# Vector Storage & Chroma

## What is a Vector Database?

Once text is converted into vectors, you need a system to store and search them efficiently.

A **vector database**:

- Stores numerical vectors
- Performs similarity search
- Finds meaning-based matches (not just keyword matches)

Example:

Search: "How do I bake bread?"  
May return documents about:
- "Making sourdough at home"
- "Beginner's guide to baking"

Even if exact words don’t match.

---

## What is Chroma?

Chroma (ChromaDB) is an open-source vector database.

It:

- Stores embeddings
- Performs fast similarity search
- Works well for RAG systems
- Can run locally
- Is commonly used in LLM applications

Other popular vector databases:

- Pinecone
- Weaviate
- FAISS (by Meta)

---

## How Embeddings + Vector DB Work Together

1. Upload documents
2. Convert text → vectors using embedding model
3. Store vectors in Chroma
4. Convert user query → vector
5. Retrieve most similar vectors
6. Send retrieved content to LLM to generate answer

---

In simple terms:

**Vector database = Stores numbers & finds similar meaning**


In [ ]:
# Pick an embedding model

# Hugging face model which is light weight and very fast but its dimension is very low(384)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# If you need use the ollama embedding model which has high dimension (768)
# embeddings = OllamaEmbeddings(model=EMBED_MODEL)

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings, collection_name="Test_docs").delete_collection()
    

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name, collection_name="Test_docs")
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [ ]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

#### Part C: Visualize!

Let visualize how the Vector DB stores the Data

In [ ]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()